# NVIDIARetriever Connector – LangChain Integration

**Motivation:** This notebook showcases the **LangChain integration** with the NVIDIA RAG Blueprint. The `NVIDIARetriever` from `langchain-nvidia-ai-endpoints` connects to the NVIDIA RAG `/v1/search` endpoint and returns standard LangChain `Document` objects, enabling seamless use in chains, agents, and RAG pipelines without custom HTTP code.

---

## Prerequisite: Run Ingestion First

**You must ingest documents before using this notebook.** Use the [ingestion_api_usage.ipynb](./ingestion_api_usage.ipynb) notebook:

1. Open [ingestion_api_usage.ipynb](./ingestion_api_usage.ipynb).
2. Execute the following cells **in order** (top to bottom):
   - **1. Install Dependencies** – `pip install aiohttp`
   - **2. Setup Base Configuration** – ingestor URL (port 8082)
   - **3. Health Check** – verify ingestor is running
   - **4. Create collection** – creates `multimodal_data` collection
   - **4. Upload Document** – FILEPATHS cell, then `upload_documents` cell
   - **5. Get Task Status** – poll until state is `FINISHED`
3. When ingestion is complete, return here and run the cells below.

Ensure the **RAG server** (port 8081) is running. See [Get Started](https://github.com/NVIDIA-AI-Blueprints/rag/blob/main/docs/deploy-docker-self-hosted.md).

---
## Setup

In [ ]:
!pip install langchain-nvidia-ai-endpoints langchain-core

import os

# RAG server URL (use collection from ingestion_api_usage.ipynb)
RAG_IPADDRESS = (
    "rag-server" if os.environ.get("AI_WORKBENCH", "false") == "true" else "localhost"
)
RAG_BASE_URL = f"http://{RAG_IPADDRESS}:8081"

# Collection from ingestion_api_usage.ipynb (default: multimodal_data)
COLLECTION_NAME = "multimodal_data"

---
## Retrieval with NVIDIARetriever

The `NVIDIARetriever` from `langchain-nvidia-ai-endpoints` connects to the NVIDIA RAG Blueprint `/v1/search` endpoint and returns LangChain `Document` objects. Use `COLLECTION_NAME` to match the collection you created in [ingestion_api_usage.ipynb](./ingestion_api_usage.ipynb).

### 2.1 Basic Sync Retrieval

In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIARetriever

retriever = NVIDIARetriever(
    base_url=RAG_BASE_URL,
    collection_names=[COLLECTION_NAME],
    k=5,
)

query = "What are the main topics or products discussed?"
docs = retriever.invoke(query)

print(f"Query: {query}")
print(f"Retrieved {len(docs)} documents:\n")
for i, doc in enumerate(docs, 1):
    content_preview = (doc.page_content or "")[:300] + "..." if len(doc.page_content or "") > 300 else (doc.page_content or "")
    score = doc.metadata.get("score", "N/A")
    source = doc.metadata.get("document_name", "N/A")
    print(f"--- Document {i} ---")
    print(f"Score: {score} | Source: {source}")
    print(f"Content: {content_preview}")
    print()

### 2.2 Custom Retrieval Parameters

In [ ]:
retriever_custom = NVIDIARetriever(
    base_url=RAG_BASE_URL,
    collection_names=[COLLECTION_NAME],
    k=3,
    vdb_top_k=50,
    enable_reranker=True,
    enable_query_rewriting=False,
)

docs = retriever_custom.invoke("Summarize key information")
print(f"Retrieved {len(docs)} documents")
for i, doc in enumerate(docs, 1):
    print(f"  {i}. {doc.metadata.get('document_name', 'N/A')} (score: {doc.metadata.get('score', 'N/A')})")

### 2.3 Async Retrieval

In [ ]:
docs = await retriever.ainvoke("What features or benefits are described?")
print(f"Async retrieval: {len(docs)} documents")
for i, doc in enumerate(docs[:3], 1):
    print(f"  {i}. {doc.metadata.get('document_name', 'N/A')}")

### 2.4 Error Handling

In [ ]:
from langchain_nvidia_ai_endpoints.retrievers import (
    NVIDIARAGConnectionError,
    NVIDIARAGServerError,
    NVIDIARAGValidationError,
)

try:
    bad_retriever = NVIDIARetriever(
        base_url="http://invalid-host:8081",
        collection_names=[COLLECTION_NAME],
    )
    bad_retriever.invoke("test")
except NVIDIARAGConnectionError as e:
    print(f"Connection error (expected): {e}")
except NVIDIARAGValidationError as e:
    print(f"Validation error: {e}")
except NVIDIARAGServerError as e:
    print(f"Server error ({e.status_code}): {e}")

print("\nError handling works as expected.")

---
## RAG Chain (Optional)

Chain `NVIDIARetriever` with `ChatNVIDIA` for end-to-end question answering. Requires `NVIDIA_API_KEY`.

In [ ]:
if os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnablePassthrough

    from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIARetriever

    retriever = NVIDIARetriever(
        base_url=RAG_BASE_URL,
        collection_names=[COLLECTION_NAME],
        k=4,
    )

    def format_docs(docs):
        return "\n\n".join(d.page_content for d in docs)

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer based only on the context below.\n\n{context}"),
        ("human", "{question}"),
    ])

    llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct")
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    answer = chain.invoke("What are the main topics or products?")
    print(answer)
else:
    print("NVIDIA_API_KEY not set. Set it to run the RAG chain with ChatNVIDIA.")

---
## Cleanup (Optional)

To remove the collection and documents, use the delete cells in [ingestion_api_usage.ipynb](./ingestion_api_usage.ipynb) (sections 7 and 9).

In [ ]:
# Use ingestion_api_usage.ipynb sections 7 (Delete Documents) and 9 (Delete Collections)
# to remove the multimodal_data collection when finished.